# E-Commerce PySpark Structured Streaming Consumer

Reads enriched order-item events from a Kafka topic and applies a multi-stage
transformation pipeline before persisting results to MongoDB via `foreachBatch`.

## Pipeline stages
1. **Ingest** – read JSON messages from Kafka
2. **Parse** – deserialize JSON payload, enforce schema, cast types
3. **Enrich** – derive computed columns (`line_total`, `event_timestamp`)
4. **Aggregation A** – windowed revenue + order count per product category (1-day tumbling window, 30-day watermark)
5. **Aggregation B** – order count per shipping country
6. **Aggregation C** – payment method distribution per window
7. **Sink** – write each micro-batch to MongoDB via `foreachBatch`

## MongoDB collections
| Collection | Description |
|---|---|
| `revenue_by_category` | Windowed category aggregates |
| `orders_by_country` | Windowed country aggregates |
| `payment_method_stats` | Windowed payment method counts |
| `orders_raw` | Enriched raw events (for validation) |

## How to run the producer
Open a terminal and run:
```bash
docker exec -it spark-notebook /bin/bash
python3 /opt/spark/work-dir/src/producers/producer.py \
    --broker kafka:9093 \
    --topic ecommerce-orders \
    --records 5000 \
    --delay 0.2
```

## 1. Create SparkSession

In [13]:
import sys
sys.path.insert(0, "/opt/spark/work-dir/src")

from SparkUtils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
)

packages = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0",
    "org.mongodb.spark:mongo-spark-connector_2.13:10.3.0",
])

su = SparkUtils(
    "ECommerceStreamingConsumer",
    master_url="local[*]",
    packages=packages,
)
su.spark.conf.set("spark.mongodb.write.connection.uri", "mongodb://mongo:27017")
su.spark

26/05/08 06:53:12 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 2. Configuration

In [14]:
KAFKA_SERVERS  = "kafka:9093"
KAFKA_TOPIC    = "ecommerce-orders"
MONGO_URI      = "mongodb://mongo:27017"
MONGO_DATABASE = "ecommerce"
CHECKPOINT_DIR = "/opt/spark/work-dir/checkpoints"

# Clean up old checkpoints so the stream starts fresh
checkpoint_path = Path(CHECKPOINT_DIR)
if checkpoint_path.exists() and checkpoint_path.is_dir():
    shutil.rmtree(checkpoint_path)
    print(f"Cleared checkpoint directory: {CHECKPOINT_DIR}")

print(f"Kafka  : {KAFKA_SERVERS}  →  topic: {KAFKA_TOPIC}")
print(f"MongoDB: {MONGO_URI}  →  db: {MONGO_DATABASE}")

Kafka  : kafka:9093  →  topic: ecommerce-orders
MongoDB: mongodb://mongo:27017  →  db: ecommerce


## 3. Schema definition

Explicit schema for the enriched order-item events produced by `producer.py`.

In [15]:
ORDER_ITEM_SCHEMA = StructType([
    StructField("order_item_id",    StringType(),  True),
    StructField("order_id",         StringType(),  True),
    StructField("product_id",       StringType(),  True),
    StructField("quantity",         IntegerType(), True),
    StructField("unit_price",       DoubleType(),  True),
    StructField("customer_id",      StringType(),  True),
    StructField("order_date",       StringType(),  True),
    StructField("total_amount",     DoubleType(),  True),
    StructField("payment_method",   StringType(),  True),
    StructField("shipping_country", StringType(),  True),
    StructField("product_name",     StringType(),  True),
    StructField("category",         StringType(),  True),
    StructField("product_price",    DoubleType(),  True),
    StructField("brand",            StringType(),  True),
    StructField("customer_name",    StringType(),  True),
    StructField("country",          StringType(),  True),
])

print(f"Schema has {len(ORDER_ITEM_SCHEMA.fields)} fields")

Schema has 16 fields


## 4. Stage 1 & 2 — Ingest + Parse

Read raw binary messages from Kafka, cast `value` to string, then parse as JSON.

In [16]:
raw_stream = (
    su.spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
    .select(F.from_json(F.col("value").cast("string"), ORDER_ITEM_SCHEMA).alias("data"))
    .select("data.*")
)

print("Kafka stream schema:")
raw_stream.printSchema()

Kafka stream schema:
root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- country: string (nullable = true)



## 5. Stage 3 — Enrich

- `line_total` — `quantity × unit_price`
- `event_timestamp` — `order_date` parsed to timestamp for windowing
- `processing_time` — wall-clock time for auditing

In [17]:
enriched = (
    raw_stream
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .withColumn("event_timestamp", F.to_timestamp(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("processing_time", F.current_timestamp())
    .filter(F.col("event_timestamp").isNotNull())
    .filter(F.col("line_total") > 0)
)

print("Enriched stream schema:")
enriched.printSchema()

Enriched stream schema:
root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- line_total: double (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- processing_time: timestamp (nullable = false)



## 6. Stage 4 — Aggregation A: Revenue by Category

1-day tumbling window grouped by `category`. 30-day watermark for late data.

In [18]:
cat_stream = (
    enriched
    .groupBy(
        F.window("event_timestamp", "1 day").alias("window"),
        F.col("category"),
    )
    .agg(
        F.sum("line_total").alias("total_revenue"),
        F.approx_count_distinct("order_id").alias("total_orders"),
        F.avg("line_total").alias("avg_line_value"),
        F.sum("quantity").alias("total_units_sold"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "category",
        F.round("total_revenue", 2).alias("total_revenue"),
        "total_orders",
        F.round("avg_line_value", 2).alias("avg_line_value"),
        "total_units_sold",
    )
)

print("revenue_by_category schema:")
cat_stream.printSchema()

revenue_by_category schema:
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- category: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- avg_line_value: double (nullable = true)
 |-- total_units_sold: long (nullable = true)



## 7. Stage 5 — Aggregation B: Orders by Country

1-day tumbling window grouped by `shipping_country`.

In [19]:
country_stream = (
    enriched
    .groupBy(
        F.window("event_timestamp", "1 day").alias("window"),
        F.col("shipping_country"),
    )
    .agg(
        F.approx_count_distinct("order_id").alias("order_count"),
        F.sum("line_total").alias("total_revenue"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "shipping_country",
        "order_count",
        F.round("total_revenue", 2).alias("total_revenue"),
    )
)

print("orders_by_country schema:")
country_stream.printSchema()

orders_by_country schema:
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- order_count: long (nullable = false)
 |-- total_revenue: double (nullable = true)



## 8. Stage 6 — Aggregation C: Payment Method Stats

1-day tumbling window grouped by `payment_method`.

In [20]:
payment_stream = (
    enriched
    .groupBy(
        F.window("event_timestamp", "1 day").alias("window"),
        F.col("payment_method"),
    )
    .agg(
        F.count("order_item_id").alias("transaction_count"),
        F.sum("total_amount").alias("total_amount"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "payment_method",
        "transaction_count",
        F.round("total_amount", 2).alias("total_amount"),
    )
)

print("payment_method_stats schema:")
payment_stream.printSchema()

payment_method_stats schema:
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- transaction_count: long (nullable = false)
 |-- total_amount: double (nullable = true)



## 9. Stage 7 — MongoDB Sink via `foreachBatch`

In [21]:
def make_mongo_writer(collection: str):
    """Returns a foreachBatch function that appends a micro-batch to MongoDB."""
    def write_batch(batch_df: DataFrame, batch_id: int) -> None:
        if batch_df.isEmpty():
            return
        row_count = batch_df.count()
        print(f"[batch {batch_id}] Writing {row_count} rows to {MONGO_DATABASE}.{collection}")
        (
            batch_df.write
            .format("mongodb")
            .mode("append")
            .option("connection.uri", MONGO_URI)
            .option("database", MONGO_DATABASE)
            .option("collection", collection)
            .save()
        )
    return write_batch


def make_raw_writer(collection: str = "orders_raw"):
    """foreachBatch writer for raw enriched events."""
    def write_batch(batch_df: DataFrame, batch_id: int) -> None:
        if batch_df.isEmpty():
            return
        row_count = batch_df.count()
        print(f"[batch {batch_id}] Writing {row_count} raw events to {MONGO_DATABASE}.{collection}")
        (
            batch_df
            .drop("processing_time")
            .write
            .format("mongodb")
            .mode("append")
            .option("connection.uri", MONGO_URI)
            .option("database", MONGO_DATABASE)
            .option("collection", collection)
            .save()
        )
    return write_batch

print("MongoDB writer functions defined.")

MongoDB writer functions defined.


In [24]:
# Minimal test: just read from Kafka and print raw values to console
debug_query = (
    su.spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9093")
    .option("subscribe", "ecommerce-orders")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as raw_value")
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 5)
    .trigger(processingTime="10 seconds")
    .start()
)

debug_query.awaitTermination(60)  # wait 60 seconds then stop


26/05/08 06:54:50 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b2cca705-4105-4a9a-8df2-f432d5178353. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+
|raw_value|
+---------+
+---------+



False

## 10. Start Streaming Queries

Four queries run in parallel, each triggered every 30 seconds.

> Interrupt the kernel to stop all queries.

In [25]:
# Temporarily disabled to isolate the issue
# q1 = (
#     cat_stream.writeStream
#     .outputMode("complete")
#     .option("checkpointLocation", f"{CHECKPOINT_DIR}/revenue_by_category")
#     .foreachBatch(make_mongo_writer("revenue_by_category"))
#     .trigger(processingTime="30 seconds")
#     .start()
# )

# q2 = (
#     country_stream.writeStream
#     .outputMode("complete")
#     .option("checkpointLocation", f"{CHECKPOINT_DIR}/orders_by_country")
#     .foreachBatch(make_mongo_writer("orders_by_country"))
#     .trigger(processingTime="30 seconds")
#     .start()
# )

# q3 = (
#     payment_stream.writeStream
#     .outputMode("complete")
#     .option("checkpointLocation", f"{CHECKPOINT_DIR}/payment_method_stats")
#     .foreachBatch(make_mongo_writer("payment_method_stats"))
#     .trigger(processingTime="30 seconds")
#     .start()
# )

q4 = (
    enriched.writeStream
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT_DIR}/orders_raw")
    .foreachBatch(make_raw_writer("orders_raw"))
    .trigger(processingTime="30 seconds")
    .start()
)

print("Streaming queries started:")
for q in [q4]:
    print(f"  [{q.id}] status: {q.status['message']}")

Streaming queries started:
  [911b8f31-7aa0-4413-a7ad-d7ccc8dd7004] status: Initializing sources


26/05/08 06:55:50 WARN StreamingQueryManager: Stopping existing streaming query [id=911b8f31-7aa0-4413-a7ad-d7ccc8dd7004, runId=8f9cc40a-e465-4852-a747-c26e90d77d58], as a new run is being started.
26/05/08 06:55:50 WARN DAGScheduler: Failed to cancel job group 8f9cc40a-e465-4852-a747-c26e90d77d58. Cannot find active jobs for it.
26/05/08 06:55:50 WARN DAGScheduler: Failed to cancel job group 8f9cc40a-e465-4852-a747-c26e90d77d58. Cannot find active jobs for it.


In [26]:
# ── Debug cell: run this while streams are active to check progress ──
import time

for i in range(5):
    print(f"\n--- Check {i+1} ---")
    for q in su.spark.streams.active:
        print(f"Query : {q.name or q.id}")
        print(f"Status: {q.status}")
        if q.lastProgress:
            print(f"Last batch id       : {q.lastProgress['batchId']}")
            print(f"Input rows/sec      : {q.lastProgress['inputRowsPerSecond']}")
            print(f"Processed rows/sec  : {q.lastProgress['processedRowsPerSecond']}")
            print(f"Num input rows      : {q.lastProgress['numInputRows']}")
        else:
            print("No progress yet")
    time.sleep(10)


--- Check 1 ---
Query : 911b8f31-7aa0-4413-a7ad-d7ccc8dd7004
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Last batch id       : 1
Input rows/sec      : 0.0
Processed rows/sec  : 0.0
Num input rows      : 0
Query : de153e03-7b17-41d9-b206-54518bb31927
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Last batch id       : 1
Input rows/sec      : 0.0
Processed rows/sec  : 0.0
Num input rows      : 0
Query : 3fbe8855-fa42-4297-a2b2-7a5756b03300
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Last batch id       : 1
Input rows/sec      : 0.0
Processed rows/sec  : 0.0
Num input rows      : 0
Query : 5ca4b5a2-1e38-43f9-9d81-467219b90e40
Status: {'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
Last batch id       : 1
Input rows/sec      : 0.0
Processed rows/sec  : 0.0
Num input rows      : 0
Que

In [27]:
# Block until a query terminates — interrupt the kernel to stop
print("Waiting for termination… (interrupt kernel to stop)")
su.spark.streams.awaitAnyTermination()

Waiting for termination… (interrupt kernel to stop)


## 11. Stop the SparkSession

In [ ]:
su.spark.stop()